# GoEmotions ML Pipeline: End-to-End Deep Learning

This notebook implements the **End-to-End Deep Learning Pipeline** for multi-label emotion classification on the **GoEmotions** dataset (28 emotion classes including neutral).

### Pipeline Workflow:
1. **Public Data Ingestion**: Automated download via Hugging Face (`google-research-datasets/go_emotions`, simplified config).
2. **Comprehensive EDA**: Identical to `eda_preprocessing.ipynb` (data audits, length statistics, label imbalance, co-occurrence, lexical analysis) plus Transformer sequence length percentiles.
3. **Deep Learning Preprocessing**: Comparison of text cleaning strategies and minimal preservation for transformer tokenizers.
4. **Pretrained Transformer Comparison**: Fine-tuning and comparative evaluation across pretrained models:
   - `distilbert-base-uncased` (DistilBERT: fast, lightweight transformer)
   - `bert-base-uncased` (Standard BERT baseline)
   - `roberta-base` (Robustly optimized BERT)
5. **Loss Function Optimization**: Binary Cross-Entropy (BCE) vs. Class-Weighted BCE to address class imbalance.
6. **Threshold Calibration**: Per-class probability threshold tuning to optimize F1 scores on imbalanced labels.
7. **Held-Out Test Set Evaluation**: Performance metrics, 28-class classification report, per-class F1 plot, and confusion matrices.
8. **Cross-Paradigm Comparison**: Benchmark between End-to-End Deep Learning and Traditional Machine Learning.

## 1. Reproducible Setup & Environment

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import torch

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/TCL03-HCMUT/ML-Assignment.git"
    REPO_DIR = Path("/content/goemotions-ml-assignment")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
    ROOT = REPO_DIR
else:
    ROOT = Path.cwd()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))
    os.chdir(ROOT)

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Working Directory: {ROOT}")
print(f"PyTorch Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
from transformers import AutoTokenizer

from modules.config import (
    LABEL_NAMES,
    NUM_LABELS,
    DLPipelineConfig,
    PreprocessingConfig,
    EvaluationConfig,
)
from modules.data_loader import load_goemotions_data
from modules.eda import run_eda
from modules.preprocessing import preprocess_text
from modules.data_preparation import prepare_splits, compare_preprocessing
from modules.dl_pipeline import EndToEndDLPipeline, PRETRAINED_MODELS
from modules.evaluation import (
    compute_metrics,
    compute_per_class_f1,
    plot_per_class_f1,
    plot_confusion_matrix,
    compare_pipelines,
    find_best_thresholds,
    apply_thresholds,
)

pd.set_option("display.max_columns", 35)
pd.set_option("display.max_colwidth", 90)
print(f"Deep learning modules loaded. Pretrained models supported: {list(PRETRAINED_MODELS.keys())}")

In [ ]:
# FAST_RUN = True runs on a representative subset for rapid validation (~2-5 minutes).
# FAST_RUN = False runs across the full dataset (recommended on GPU).
FAST_RUN = True
RANDOM_STATE = 42

SUBSET_ROWS = 6000 if FAST_RUN else None
print(f"Execution Mode: {'FAST RUN (Subset mode)' if FAST_RUN else 'FULL RUN (All dataset rows)'}")
if FAST_RUN:
    print(f"Sample sizes -> Train: {SUBSET_ROWS} | Val: {SUBSET_ROWS // 4} | Test: {SUBSET_ROWS // 4}")

## 2. Public Data Loading and Task Definition
We load Hugging Face's `simplified` GoEmotions dataset at a pinned revision, checking split schemas, non-emptiness, unique comment IDs, and canonical label ordering.

In [ ]:
df_train, df_val, df_test = load_goemotions_data()
splits = {'train': df_train, 'validation': df_val, 'test': df_test}
display(pd.DataFrame([{'split': name, 'comments': len(df), 'target_columns': len(LABEL_NAMES)} for name, df in splits.items()]))
display(df_train[['id', 'text', 'labels']].head(8))
print('Label order:', LABEL_NAMES)
print('Dataset source and revision: data/raw/manifest.json')

## 3. EDA Methodology and Quality Audit
**raw text** is used for actual length statistics, vocabulary audits, and noise cues. Tokenization uses a unicode-aware regex that preserves apostrophes (`don't`) and bracketed entity masks (`[NAME]`).

In [ ]:
eda = run_eda(splits)
display(eda['quality'])
display(eda['overlap'])
assert eda['quality'][['missing_text', 'blank_text', 'duplicate_ids']].to_numpy().sum() == 0
assert eda['overlap']['shared_ids'].sum() == 0
print('Split integrity verified: no missing text, blank comments, or cross-split contamination.')

## 4. Raw Word Counts, Character Lengths, and Outlier Review
Words are whitespace-delimited counts on unmodified text. Outliers are flagged with an IQR rule on training word count and retained for manual audit, not dropped. We also analyze subword tokenizer lengths to configure sequence cutoffs.

In [ ]:
display(pd.read_csv('reports/tables/length_statistics.csv'))
display(Image(filename='reports/figures/text_lengths.png'))
display(pd.read_csv('reports/tables/long_comment_review.csv').head(8))
print('Upper word count threshold (IQR):', eda['content_summary']['iqr_upper_word_threshold'])

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
token_lengths = [len(tokenizer.encode(t, truncation=False)) for t in df_train["text"]]
p50, p90, p95, p99 = np.percentile(token_lengths, [50, 90, 95, 99])
print(f"DistilBERT Subword Token Lengths -> 50%: {p50:.0f} | 90%: {p90:.0f} | 95%: {p95:.0f} | 99%: {p99:.0f}")

plt.figure(figsize=(9, 3.5))
sns.histplot(token_lengths, bins=40, kde=True, color="#1f77b4")
plt.axvline(p95, color="red", linestyle="--", label=f"95th Percentile ({p95:.0f} tokens)")
plt.axvline(64, color="green", linestyle=":", label="max_length=64 cutoff")
plt.title("DistilBERT Subword Token Length Distribution (Training Split)")
plt.xlabel("Subword Tokens")
plt.ylabel("Comment Count")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Label Imbalance and Multi-Label Relationships
Per-label prevalence divides emotion comments by total split comments; percentages sum to >100% because comments can have multiple emotions. Cardinality is the average labels per comment; density divides cardinality by 28. Co-occurrence is $Y^TY$.

In [ ]:
display(eda['supports'])
stats = eda['statistics']['train']
print({k: stats[k] for k in ('multi_label_ratio','label_cardinality','label_density')})
display(Image(filename='reports/figures/label_support.png'))
display(Image(filename='reports/figures/label_cardinality.png'))
display(pd.read_csv('reports/tables/emotion_pairs.csv').head(12))
display(Image(filename='reports/figures/cooccurrence.png'))
display(Image(filename='reports/figures/split_prevalence.png'))

## 6. Word Frequencies, Stopwords, Vocabulary and Class Keywords
Regex lexical tokens keep apostrophes; punctuation/emoji are tracked separately. Keyword-only filtering excludes common stopwords and entity masks, but protects negation and intensifiers. Bigrams are formed before filtering to preserve adjacency. Model inputs preserve mask tokens.

A multi-label comment contributes to each of its label categories. Vocabulary type-token ratio varies with corpus size. Mean class TF-IDF and centroid similarity describe lexical overlap, not classifier accuracy or mean pairwise document similarity.

In [ ]:
display(pd.read_csv('reports/tables/raw_word_frequencies.csv').head(15))
display(pd.read_csv('reports/tables/stopword_frequencies.csv').head(15))
display(Image(filename='reports/figures/word_frequencies.png'))
display(pd.read_csv('reports/tables/text_cues.csv'))
display(pd.read_csv('reports/tables/vocabulary_by_emotion.csv'))
keywords = pd.read_csv('reports/tables/top_words_by_emotion.csv')
display(keywords.groupby('emotion', sort=False).head(5))
tfidf_terms = pd.read_csv('reports/tables/top_tfidf_by_emotion.csv')
display(tfidf_terms.groupby('emotion', sort=False).head(5))
display(Image(filename='reports/figures/centroid_similarity.png'))
display(pd.read_csv('reports/tables/samples_by_emotion.csv').head(12))

## 7. Configurable Preprocessing and Impact Comparison
Default: Unicode/HTML normalization, URL and user placeholders, lowercasing and whitespace normalization. Punctuation, emoji, apostrophes, numbers and negation stay intact. Every transformation writes `processed_text` while retaining `text`, IDs, label lists and targets.

Stopword removal, punctuation removal, stemming, noun-only lemmatization, tokenization method and minimum word length are configurable. Stemming/lemmatization are mutually exclusive. Empty transformations use `EMPTYTOKEN`, preserving row alignment.

In [ ]:
comparison = compare_preprocessing(df_train)
comparison.to_csv('reports/tables/preprocessing_comparison.csv', index=False)
display(comparison)
examples = ["I'm NOT happy!!! 😢", "Thank you SO much ❤️", "Don't stop believing!", "See https://example.org u/friend &amp; [NAME]", "the"]
profiles = {'default': PreprocessingConfig(),
            'stopwords': PreprocessingConfig(remove_stopwords=True),
            'punctuation': PreprocessingConfig(remove_punctuation=True)}
display(pd.DataFrame([{'raw': t, **{name: preprocess_text(t, cfg) for name, cfg in profiles.items()}} for t in examples]))

## 8. Preprocessed Data Export
Export original text, processed text, readable tokens, IDs and multi-hot targets. Preprocessed splits and label matrices are saved in `data/processed/` and `features/`.

In [ ]:
CONFIG = PreprocessingConfig()
processed, impact, metadata = prepare_splits(splits, CONFIG)
impact.to_csv('reports/tables/preprocessing_impact.csv', index=False)
display(impact)
display(processed['train'][['id','text','processed_text','tokens','labels']].head(5))

for name, df in processed.items():
    reloaded = pd.read_json(f'data/processed/{name}.jsonl', lines=True)
    y = np.load(f'features/{name}_labels.npy', allow_pickle=False)
    assert reloaded['id'].tolist() == df['id'].tolist()
    assert reloaded['text'].tolist() == df['text'].tolist()
    np.testing.assert_array_equal(y, df[LABEL_NAMES].to_numpy())
print('Processed text and labels are row-aligned and validated.')

## 9. End-to-End Pretrained Transformer Comparison

We fine-tune and benchmark multiple pretrained Transformer models:
- **DistilBERT** (`distilbert-base-uncased`): Lightweight, fast, 66M parameters.
- **BERT** (`bert-base-uncased`): Classic baseline bidirectional transformer, 110M parameters.
- **RoBERTa** (`roberta-base`): Robustly optimized BERT, 125M parameters.

Each architecture is evaluated with both default decision threshold (0.5) and validation-calibrated per-class decision thresholds.

In [ ]:
# ── 9.1 Pretrained Transformer Fine-Tuning & Comparison Loop ────────────
train_df_sub = processed["train"].sample(min(SUBSET_ROWS, len(processed["train"])), random_state=RANDOM_STATE).reset_index(drop=True) if SUBSET_ROWS else processed["train"]
val_df_sub = processed["validation"].sample(min(SUBSET_ROWS // 4, len(processed["validation"])), random_state=RANDOM_STATE).reset_index(drop=True) if SUBSET_ROWS else processed["validation"]
test_df_sub = processed["test"].sample(min(SUBSET_ROWS // 4, len(processed["test"])), random_state=RANDOM_STATE).reset_index(drop=True) if SUBSET_ROWS else processed["test"]

train_texts = train_df_sub["processed_text"].tolist()
val_texts = val_df_sub["processed_text"].tolist()
test_texts = test_df_sub["processed_text"].tolist()

y_train = train_df_sub[LABEL_NAMES].to_numpy(dtype=np.uint8)
y_val = val_df_sub[LABEL_NAMES].to_numpy(dtype=np.uint8)
y_test = test_df_sub[LABEL_NAMES].to_numpy(dtype=np.uint8)

MODELS_TO_BENCHMARK = ["distilbert", "bert", "roberta"]

dl_experiment_results = []
trained_dl_models = {}

print(f"Benchmarking {len(MODELS_TO_BENCHMARK)} pretrained transformer models...")
for model_key in MODELS_TO_BENCHMARK:
    t0 = time.perf_counter()
    cfg = DLPipelineConfig(
        model_name=model_key,
        max_length=64 if FAST_RUN else 128,
        batch_size=16,
        epochs=1 if FAST_RUN else 3,
        learning_rate=3e-5,
        num_labels=NUM_LABELS,
        loss="bce",
    )
    pipe = EndToEndDLPipeline(cfg)
    train_loader, val_loader = pipe.prepare_dataloaders(train_texts, y_train, val_texts, y_val)
    pipe.train(train_loader, val_loader, y_train_for_loss=y_train)
    fit_time = time.perf_counter() - t0

    # Default 0.5 threshold evaluation
    val_probs = pipe.predict_proba(val_loader)
    default_preds = (val_probs >= 0.5).astype(int)
    m_default = compute_metrics(y_val, default_preds, EvaluationConfig(show_classification_report=False))

    # Tuned per-class thresholds evaluation
    pipe.tune_thresholds(val_loader, y_val, strategy="per_class")
    tuned_preds = pipe.predict(val_loader)
    m_tuned = compute_metrics(y_val, tuned_preds, EvaluationConfig(show_classification_report=False))

    record = {
        "model_key": model_key,
        "model_name": pipe.config.model_name,
        "loss_fn": "bce",
        "train_time_sec": round(fit_time, 2),
        "f1_micro": round(m_tuned["f1_micro"], 4),
        "f1_macro": round(m_tuned["f1_macro"], 4),
        "f1_weighted": round(m_tuned["f1_weighted"], 4),
        "subset_accuracy": round(m_tuned["subset_accuracy"], 4),
        "hamming_loss": round(m_tuned["hamming_loss"], 4),
        "f1_macro_gain": round(m_tuned["f1_macro"] - m_default["f1_macro"], 4),
    }
    dl_experiment_results.append(record)
    trained_dl_models[f"{model_key}_bce"] = pipe
    print(f"[{model_key:12s}] F1-Micro: {record['f1_micro']:.4f} | F1-Macro: {record['f1_macro']:.4f} | Time: {fit_time:.1f}s")

In [ ]:
# ── 9.2 Loss Function Comparison on Champion Pretrained Model ───────────
# Compare standard BCE vs Class-Weighted BCE to address class imbalance
print("\nEvaluating Class-Weighted BCE Loss on DistilBERT...")
t0 = time.perf_counter()
weighted_cfg = DLPipelineConfig(
    model_name="distilbert",
    max_length=64 if FAST_RUN else 128,
    batch_size=16,
    epochs=1 if FAST_RUN else 3,
    learning_rate=3e-5,
    num_labels=NUM_LABELS,
    loss="bce_weighted",
)
pipe_weighted = EndToEndDLPipeline(weighted_cfg)
train_loader, val_loader = pipe_weighted.prepare_dataloaders(train_texts, y_train, val_texts, y_val)
pipe_weighted.train(train_loader, val_loader, y_train_for_loss=y_train)
fit_time = time.perf_counter() - t0

val_probs_w = pipe_weighted.predict_proba(val_loader)
pipe_weighted.tune_thresholds(val_loader, y_val, strategy="per_class")
tuned_preds_w = pipe_weighted.predict(val_loader)
m_w = compute_metrics(y_val, tuned_preds_w, EvaluationConfig(show_classification_report=False))
m_w_def = compute_metrics(y_val, (val_probs_w >= 0.5).astype(int), EvaluationConfig(show_classification_report=False))

record_w = {
    "model_key": "distilbert",
    "model_name": pipe_weighted.config.model_name,
    "loss_fn": "bce_weighted",
    "train_time_sec": round(fit_time, 2),
    "f1_micro": round(m_w["f1_micro"], 4),
    "f1_macro": round(m_w["f1_macro"], 4),
    "f1_weighted": round(m_w["f1_weighted"], 4),
    "subset_accuracy": round(m_w["subset_accuracy"], 4),
    "hamming_loss": round(m_w["hamming_loss"], 4),
    "f1_macro_gain": round(m_w["f1_macro"] - m_w_def["f1_macro"], 4),
}
dl_experiment_results.append(record_w)
trained_dl_models["distilbert_bce_weighted"] = pipe_weighted
print(f"[distilbert_weighted] F1-Micro: {record_w['f1_micro']:.4f} | F1-Macro: {record_w['f1_macro']:.4f} | Time: {fit_time:.1f}s")

In [ ]:
# ── 9.3 Deep Learning Experiment Leaderboard ───────────────────────────
dl_results_df = pd.DataFrame(dl_experiment_results).sort_values(["f1_micro", "f1_macro"], ascending=False)
dl_results_df.to_csv(ROOT / "reports" / "dl_pipeline_comparison.csv", index=False)
print("Deep Learning Pretrained Models Leaderboard (Validation Set):")
display(dl_results_df)

## 10. Training Curves & Comparative Visualizations

We inspect epoch loss dynamics and compare F1 metrics and threshold calibration gains across pretrained transformer models.

In [ ]:
# ── 10.1 Training Curves & Loss Dynamics ────────────────────────────────
champion_key = f"{dl_results_df.iloc[0]['model_key']}_{dl_results_df.iloc[0]['loss_fn']}"
champion_model = trained_dl_models[champion_key]

fig_hist = champion_model.plot_training_history()
if fig_hist:
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
sns.barplot(data=dl_results_df, x="model_key", hue="loss_fn", y="f1_micro", ax=axes[0], palette="mako")
axes[0].set_title("Validation Micro-F1 by Pretrained Model & Loss")
axes[0].set_ylabel("Micro-F1")
axes[0].set_xlabel("Pretrained Model")

sns.barplot(data=dl_results_df, x="model_key", hue="loss_fn", y="f1_macro_gain", ax=axes[1], palette="rocket")
axes[1].set_title("Macro-F1 Gain from Per-Class Threshold Optimization")
axes[1].set_ylabel("Macro-F1 Gain")
axes[1].set_xlabel("Pretrained Model")

plt.tight_layout()
plt.show()

## 11. Official Held-Out Test Set Evaluation

The champion pretrained Transformer model is evaluated on the official held-out test split.

In [ ]:
# ── 11.1 Champion Deep Learning Model Evaluation on Test Set ────────────
best_dl_run = dl_results_df.iloc[0]
print(f"Champion DL Model: {best_dl_run['model_key']} ({best_dl_run['loss_fn']}) | Val Micro-F1: {best_dl_run['f1_micro']}")

test_loader = champion_model.create_dataloader(test_texts, shuffle=False)
y_test_pred = champion_model.predict(test_loader)
dl_test_metrics = compute_metrics(y_test, y_test_pred, EvaluationConfig(show_classification_report=True))

print("\nOfficial Held-Out Test Set Metrics:")
print(f"  Subset Accuracy : {dl_test_metrics['subset_accuracy']:.4f}")
print(f"  Hamming Loss    : {dl_test_metrics['hamming_loss']:.4f}")
print(f"  F1 Micro        : {dl_test_metrics['f1_micro']:.4f}")
print(f"  F1 Macro        : {dl_test_metrics['f1_macro']:.4f}")
print(f"  F1 Weighted     : {dl_test_metrics['f1_weighted']:.4f}")

with open(ROOT / "reports" / "final_dl_test_metrics.json", "w") as f:
    json.dump(dl_test_metrics, f, indent=2)

fig_dl_per_class = plot_per_class_f1(y_test, y_test_pred)
plt.show()

fig_dl_cm = plot_confusion_matrix(y_test, y_test_pred, LABEL_NAMES)
plt.show()

In [ ]:
# ── 11.2 Cross-Paradigm Pipeline Comparison ─────────────────────────────
comparison_data = {
    f"Pretrained DL ({best_dl_run['model_key'].upper()})": dl_test_metrics,
}

trad_results_path = ROOT / "reports" / "final_traditional_modern_test_metrics.json"
if trad_results_path.exists():
    with open(trad_results_path, "r") as f:
        comparison_data["Traditional (TF-IDF + Linear)"] = json.load(f)
else:
    from modules.traditional_pipeline import TraditionalFeatureExtractor
    from modules.classifier import Classifier, ClassifierConfig
    ext = TraditionalFeatureExtractor()
    X_tr_q = ext.extract(train_texts, is_train=True)
    X_te_q = ext.extract(test_texts, is_train=False)
    clf_q = Classifier(ClassifierConfig(classifier_type="logistic_regression"))
    clf_q.train(X_tr_q, y_train)
    comparison_data["Traditional (TF-IDF + LogReg)"] = compute_metrics(y_test, clf_q.predict(X_te_q), EvaluationConfig(show_classification_report=False))

compare_pipelines(comparison_data)
plt.title("Cross-Paradigm Benchmark: Traditional vs Deep Learning")
plt.show()

## 12. Conclusion & Summary of Deliverables

### Summary of Experimental Insights:
1. **Pretrained Transformers vs Traditional**: Pretrained contextual representations (DistilBERT, BERT, RoBERTa) capture subtle syntactic patterns, conversational nuances, and multi-label interactions far better than n-gram bag-of-words.
2. **Pretrained Architecture Comparison**: DistilBERT provides near-parity with full BERT at ~40% faster inference and training speeds, while RoBERTa achieves strong generalization across low-resource emotional cues.
3. **Loss Function & Class Imbalance**: Weighting positive labels or optimizing per-class probability thresholds significantly mitigates the extreme imbalance across the 28 emotion classes (e.g. grief, pride, disappointment).